# VectorRS: End-to-End High-Performance Benchmark Suite

This notebook automates the complete execution and visualization of the **6 publication-grade benchmark suites** for **VectorRS** on Kaggle (configured for **2x NVIDIA T4 GPUs** and multi-core CPU).

### Benchmark Execution Order:
1. **CPU SIMD Intrinsics Acceleration** (`simd_benchmark.py`)
2. **Approximate Nearest Neighbor (ANN) Query Search** (`query_benchmark.py`)
3. **Shared-Memory Multi-Core Index Construction** (`index_benchmark.py`)
4. **NVIDIA CUDA GPU Hardware Acceleration & DDP** (`cuda_benchmark.py`)
5. **Multi-Mechanism k-Means & CUDA-Aware MPI** (`mpi_benchmark.py`)
6. **Distributed Cluster Scatter-Gather Scaling** (`worker_benchmark.py`)

## 1. Repository Setup
Clean previous working directory, clone fresh repository, and enter the project root.

In [ ]:
"""Kaggle Benchmark Execution Script for VectorRS."""

import os

from IPython.display import SVG, display

# Remove old directory if exists
!rm -rf vector-rs

# Clone the repository
!git clone https://github.com/Malset2603/vector-rs.git

# Change working directory to the cloned repository
%cd vector-rs

## 2. Environment & Toolchain Setup
Install Rust toolchain (edition 2024 / 1.80+), Protocol Buffers compiler (`protoc`), OpenMPI headers, and verify dual NVIDIA GPU allocation.

In [ ]:
# Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

# Add cargo to PATH
os.environ["PATH"] = f"{os.path.expanduser('~')}/.cargo/bin:" + os.environ["PATH"]
os.environ["RUSTFLAGS"] = "-C target-cpu=native"

# Install Protobuf and OpenMPI system dependencies
!apt-get update -qq && apt-get install -y -qq protobuf-compiler libopenmpi-dev openmpi-bin

# Verify toolchains
!rustc --version && cargo --version
!protoc --version
!nvidia-smi

## 3. Build Rust Workspace in Release Mode
Compiles all crates (`vector-coordinator`, `vector-worker`, `vector-index`, `vector-simd`, `vector-cuda`, `vector-mpi`, `vector-proto`) with maximum optimizations (`-O3`).

In [ ]:
!cargo build --release --all-targets

## 4. Helper Function for Inline SVG Display

In [ ]:
def show_plot(svg_filename: str) -> None:
    """Displays generated SVG benchmark plot inline within the notebook."""
    svg_path = f"scripts/benchmarks/{svg_filename}"
    if os.path.exists(svg_path):
        display(SVG(filename=svg_path))
    else:
        print(f"[ERROR] File not found: {svg_path}")

## Benchmark 1: CPU SIMD Intrinsics Acceleration
Evaluates scalar baseline vs. AVX2 + FMA 256-bit register vectorization across 10 distance and similarity metrics.

In [ ]:
!python scripts/benchmarks/simd_benchmark.py
show_plot("simd_benchmark.svg")

## Benchmark 2: Approximate Nearest Neighbor (ANN) Query Search
Evaluates query search latency of HNSW Graph and IVF-PQ Index against exact Flat brute-force baseline.

In [ ]:
!python scripts/benchmarks/query_benchmark.py
show_plot("query_benchmark.svg")

## Benchmark 3: Shared-Memory Multi-Core Index Construction
Measures Rayon work-stealing multi-core strong scaling during HNSW graph and IVF-PQ centroids construction.

In [ ]:
!python scripts/benchmarks/index_benchmark.py
show_plot("index_benchmark.svg")

## Benchmark 4: NVIDIA CUDA GPU Hardware Acceleration & DDP
Evaluates speedup across CPU Baseline vs. 1x GPU CUDA vs. 2x GPU DDP (Distributed Data Parallel) with asynchronous CUDA streams.

In [ ]:
!python scripts/benchmarks/cuda_benchmark.py
show_plot("cuda_benchmark.svg")

## Benchmark 5: Multi-Mechanism k-Means & CUDA-Aware MPI
Compares clustering duration across: Without MPI (Single CPU), MPI-CPU (Distributed), and CUDA-Aware MPI (1x GPU & 2x GPUs with Direct VRAM-to-NIC DMA).

In [ ]:
!python scripts/benchmarks/mpi_benchmark.py
show_plot("mpi_benchmark.svg")

## Benchmark 6: Distributed Cluster Scatter-Gather Scaling
Measures end-to-end cluster search throughput (QPS) and query latency scaling as worker shard nodes scale horizontally.

In [ ]:
!python scripts/benchmarks/worker_benchmark.py
show_plot("worker_benchmark.svg")